# 04 · Discretisation

Turns cleaned numbers into items such as `CHL_COP_1=[4,6)` so Apriori can count them (paper Section 3.3, step 2).

1. **Design breakpoints** from engineering limits (see `DESIGN_BREAKPOINTS` in `src/discretise.py`).
2. **Entropy refinement** (paper Eq. 7, η = 1.5 %): inside each design interval, keep a split only if it reduces the entropy of the **efficiency band** (plant COP low / medium / high) by at least 1.5 %.
3. Variables with neither get **tertiles** (low / medium / high); on/off columns become `on` / `off`.

Breakpoints are learned on the **fault-free** file only and applied unchanged to every fault file.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src import discretise as dz

S = dz.load_settings("config.yaml")
pd.set_option("display.width", 200, "display.max_colwidth", 120)
S


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



{'eta': 0.015,
 'min_frac': 0.05,
 'max_bins': 6,
 'n_grid': 40,
 'target': 'PLANT_COP',
 'target_bins': 3,
 'decimals': 1}

## Load the cleaned fault-free data
Uses Member A's output in `data/processed/`. If it is still 1-minute data, it is averaged to 15 minutes here and only rows with at least one chiller on are kept.

In [2]:
def load_processed(stem):
    for ext, reader in [(".parquet", pd.read_parquet), (".pkl", pd.read_pickle)]:
        p = Path("data/processed") / f"{stem}{ext}"
        if p.exists():
            return reader(p)
    raise FileNotFoundError(f"data/processed/{stem}.parquet or .pkl not found - run the preprocessing notebook first")

def to_15min(df):
    step = df.index.to_series().diff().median()
    if step < pd.Timedelta("15min"):
        df = df.resample("15min").mean()
        sta = [c for c in df.columns if c.endswith(tuple(f"_STA_{i}" for i in range(1, 4)))]
        df[sta] = (df[sta] > 0.5).astype(float)          # status: on if on most of the 15 min
    chl = [c for c in df.columns if c.startswith("CHL_STA_")]
    return df[df[chl].max(axis=1) > 0.5]                 # keep rows with a chiller on

ff = to_15min(load_processed("ChillerPlant"))
print(ff.shape)

(32978, 94)


## Fit the breakpoints (fault-free year only)

In [3]:
specs = dz.fit_breakpoints(ff, s=S)
summary = dz.describe(specs, ff)
print(len(specs), "variables discretised")
summary["method"].value_counts()

93 variables discretised


method
entropy             61
design + entropy    14
tertiles            10
binary               8
Name: count, dtype: int64

## Five example variables
- `CHL_SW_TEMP_1`: design limits 44 / 54 °F (the plant's measured set-point reset range)
- `CT_APPROACH_1`: design approach 8 °F
- `CHL_COP_1`: no design limit (this plant runs at COP ≈ 2.5) → entropy splits or tertiles
- `OA_TEMP_WB`: no design limit → entropy splits or tertiles
- `CWL_SEC_PM_SPD_1`: pump speed (0–1), no design limit

In [4]:
examples = ["CHL_SW_TEMP_1", "CT_APPROACH_1", "CHL_COP_1", "OA_TEMP_WB", "CWL_SEC_PM_SPD_1"]
summary[summary["variable"].isin(examples)].set_index("variable").loc[[e for e in examples if e in specs]]

,method,breakpoints,shares
variable,,,
CHL_SW_TEMP_1,design + entropy,"[44.0, 48.7, 51.1, 53.4, 54.0]","[51.1,53.4): 6%, [53.4,54): 62%, >=54: 4%, [48.7,51.1): 8%, [44,48.7): 15%, <44: 5%"
CT_APPROACH_1,design + entropy,"[8.0, 9.8]",">=9.8: 7%, [8,9.8): 49%, <8: 45%"
CHL_COP_1,entropy,"[1.9, 2.9, 3.1]",">=3.1: 10%, [2.9,3.1): 11%, [1.9,2.9): 72%, <1.9: 7%"
OA_TEMP_WB,entropy,"[51.9, 58.4, 60.8, 67.6, 74.5]","<51.9: 59%, [51.9,58.4): 12%, [58.4,60.8): 4%, [60.8,67.6): 15%, [67.6,74.5): 9%, >=74.5: 2%"
CWL_SEC_PM_SPD_1,entropy,"[0.6, 0.7, 0.8, 1.0]","[0.8,1): 37%, >=1: 18%, [0.7,0.8): 8%, [0.6,0.7): 4%, <0.6: 33%"


## All variables

In [5]:
summary

,variable,method,breakpoints,shares
0,CDWL_CW_FLOW,entropy,[643.2],"<643.2: 99%, >=643.2: 1%"
1,CDWL_PM_POW_1,entropy,[46.5],"<46.5: 99%, >=46.5: 1%"
2,CDWL_PM_POW_2,entropy,[43.8],"<43.8: 87%, >=43.8: 13%"
3,CDWL_PM_POW_3,tertiles,[0.0],>=0: 100%
4,CDWL_RW_TEMP,entropy,"[69.6, 73.4, 91.4]","<69.6: 74%, [69.6,73.4): 5%, [73.4,91.4): 20%, >=91.4: 1%"
...,...,...,...,...
88,CT_APPROACH_1,design + entropy,"[8.0, 9.8]",">=9.8: 7%, [8,9.8): 49%, <8: 45%"
89,CT_APPROACH_2,design + entropy,"[8.0, 9.2]","<8: 52%, [8,9.2): 42%, >=9.2: 6%"
90,CT_APPROACH_3,design + entropy,"[6.3, 7.5, 8.0, 9.0, 9.9]","[6.3,7.5): 23%, [8,9): 21%, <6.3: 12%, [7.5,8): 15%, >=9.9: 11%, [9,9.9): 18%"
91,PLANT_POW,entropy,"[172.8, 189.9, 233.2, 270.0, 300.4]","[233.2,270): 2%, [189.9,233.2): 6%, [172.8,189.9): 3%, <172.8: 72%, [270,300.4): 2%, >=300.4: 14%"


## Save the breakpoints (used for every file)

In [6]:
out = Path("results/breakpoints.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(specs, indent=2, default=float))
print("saved", out)

saved results\breakpoints.json


## Transactions

In [7]:
tx = dz.to_transactions(dz.discretise(ff, specs))
print("transactions:", len(tx))
print("items per transaction (mean):", round(np.mean([len(t) for t in tx]), 1))
print("distinct items:", len(set().union(*tx)))
sorted(tx[len(tx) // 2])[:15]

transactions: 32978
items per transaction (mean): 83.4
distinct items: 379


['CDWL_CW_FLOW=<643.2',
 'CDWL_PM_POW_1=<46.5',
 'CDWL_PM_POW_2=<43.8',
 'CDWL_PM_POW_3=>=0',
 'CDWL_RW_TEMP=[73.4,91.4)',
 'CDWL_SW_TEMP=[69.7,73.2)',
 'CHL_CDDT_1=>=8.3',
 'CHL_CD_FLOW_1=<321.6',
 'CHL_CD_FLOW_2=<321.6',
 'CHL_CD_FLOW_3=>=0',
 'CHL_COMP_SPD_CTRL_1=[0.6,0.9)',
 'CHL_COMP_SPD_CTRL_2=<0.3',
 'CHL_COMP_SPD_CTRL_3=>=0',
 'CHL_COOL_1=>=337.5',
 'CHL_COOL_2=<115.2']

## Same breakpoints on a fault file (consistency check)

In [8]:
fault = to_15min(load_processed("ChillerPlant_coolingtower_fouling_065"))
d_ff = dz.discretise(ff, specs)["CT_APPROACH_1"].value_counts(normalize=True).rename("fault-free")
d_ft = dz.discretise(fault, specs)["CT_APPROACH_1"].value_counts(normalize=True).rename("tower fouling 65%")
pd.concat([d_ff, d_ft], axis=1).round(3)   # fouling should push CT_APPROACH_1 into the higher intervals

,fault-free,tower fouling 65%
CT_APPROACH_1,,
"[8,9.8)",0.486,0.520
<8,0.446,0.172
>=9.8,0.069,0.308
